<a href="https://colab.research.google.com/github/KumudithaSilva/llama3-domain-adaptation/blob/feature-base-fine-tuning/llama3_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Llama3 Fine-Tuning

## Import Libraries

In [1]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 31.5 MB/s eta 0:00:00


In [2]:
!wget -q https://raw.githubusercontent.com/KumudithaSilva/llama3-domain-adaptation/feature-base-model/evaluator.py -O evaluator.py

In [3]:
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login

import torch
import transformers
from datasets import load_dataset, Dataset, DatasetDict
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed, BitsAndBytesConfig

# MLOps platform used for experiment tracking
import wandb

# Parameter-Efficient Fine-tuning library
from peft import LoraConfig, PeftConfig

# The Transformer Reinforcement Learning library with
# SFT (Supervised Fine-Tuning) and SFTConfig (Supervised Fine-Tuning Config)
from trl import SFTTrainer, SFTConfig

from datetime import  datetime
import matplotlib.pyplot as plt

## Load Dataset From HuggingFace

In [4]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

PROJECT_NAME = "stream_price"
TASK = "fine-tuning"

DATA_USER = "KumudithaSilva"
DATASET_NAME = f"{DATA_USER}/stream_items_prompt_lite"

RUN_NAME =  f"{TASK}-{datetime.now():%Y-%m-%d_%H.%M.%S}"

PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{DATA_USER}/{PROJECT_RUN_NAME}"

In [13]:
print(f"DATASET_NAME: {DATASET_NAME}")
print(f"RUN_NAME: {RUN_NAME} \n")
print(f"PROJECT_RUN_NAME: {PROJECT_RUN_NAME}")
print(f"HUB_MODEL_NAME: {HUB_MODEL_NAME}")

DATASET_NAME: KumudithaSilva/stream_items_prompt_lite
RUN_NAME: fine-tuning-2026-04-30_11.01.12 

PROJECT_RUN_NAME: stream_price-fine-tuning-2026-04-30_11.01.12
HUB_MODEL_NAME: KumudithaSilva/stream_price-fine-tuning-2026-04-30_11.01.12


In [14]:
# Hyper-parameters

EPOCHS = 1
BATCH_SIZE = 32
MAX_SEQUENCE_LENGTH = 170
GRADIENT_ACCUMULATION_STEPS = 1

In [15]:
# Hyper-parameters - QLoRA

QUANT_4_BIT = True

LORA_R = 128
LORA_ALPHA = LORA_R * 2

ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]

TARGET_MODULES = ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

In [16]:
# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

In [17]:
# Tracking

VAL_SIZE = 500
LOG_STEPS = 5
SAVE_STEPS = 20 # 100
LOG_TO_WANDB = True

In [19]:
# This works on A100 GPU
use_bf16

False

## Log in to HuggingFace and Weights & Biases

In [18]:
hf_token = userdata.get('HUGGING_KEY')
login(hf_token)

In [20]:
# Log in to Weights & Biases
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: kumudithasilva66 (kumudithasilva66-self) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [21]:
# Configure Weights & Biases to record the project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

In [22]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

## Load Dataset From HuggingFace

In [23]:
dataset = load_dataset(DATASET_NAME)

train = dataset['train'].remove_columns(['id'])
val = dataset['validation'].remove_columns(['id'])
test = dataset['test'].remove_columns(['id'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/563 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/262k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/264k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17600 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2200 [00:00<?, ? examples/s]

## Load Llama Model

### Quantization

In [24]:
# Quantization config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
    )

### Tokenizer

In [25]:
# Tokenizer config
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

### Base Model

In [26]:
# Base model config
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    )

base_model.generation_config.pad_token_id = tokenizer.pad_token_id

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

### Memory Footprint

In [27]:
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

Memory footprint: 2.2 GB


In [28]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNo

## LoRA Weights and Dimensions Explanation

In [29]:
# Each target module has 2 LoRA adaptor matrices:
#   lora_A and lora_B
#
# They do NOT add matrices directly.
# Instead, they form a low-rank update:
#
#   ΔW = α * (A @ B)
#
# where:
#   A: (out_features × r)
#   B: (r × in_features)
#
# So parameter count is:
#   A params + B params
#   NOT matrix addition

r = 256

# -------------------------
# Attention layers (LoRA params)
# -------------------------

# q_proj: 3072 → 3072
lora_q_proj = (3072 * r) + (r * 3072)

# k_proj: 3072 → 1024
lora_k_proj = (3072 * r) + (r * 1024)

# v_proj: 3072 → 1024
lora_v_proj = (3072 * r) + (r * 1024)

# o_proj: 3072 → 3072
lora_o_proj = (3072 * r) + (r * 3072)

# -------------------------
# MLP layers (LoRA params)
# -------------------------

# gate_proj: 3072 → 8192
lora_gate_proj = (3072 * r) + (r * 8192)

# up_proj: 3072 → 8192
lora_up_proj = (3072 * r) + (r * 8192)

# down_proj: 8192 → 3072
lora_down_proj = (8192 * r) + (r * 3072)

# -------------------------
# Total per transformer layer
# -------------------------

lora_layer = (
    lora_q_proj +
    lora_k_proj +
    lora_v_proj +
    lora_o_proj +
    lora_gate_proj +
    lora_up_proj +
    lora_down_proj
)

# -------------------------
# Full model (28 layers)
# -------------------------

num_layers = 28
params = lora_layer * num_layers

# -------------------------
# Memory (FP32 = 4 bytes per param)
# -------------------------

size_mb = (params * 4) / 1_000_000

print(f"Total LoRA params: {params:,}")
print(f"Approx size: {size_mb:,.1f} MB")

Total LoRA params: 389,021,696
Approx size: 1,556.1 MB


After fine-tuning LLaMA 3.2 with LoRA:

1. The base model remains the same (`base_model` is unchanged).

2. The token embedding layer is unchanged:  
   `embed_tokens: Embedding (128256, 3072)`

3. The model contains 28 decoder layers (Transformer blocks). Each layer includes self-attention with projection matrices:  
   `q_proj, k_proj, v_proj, o_proj`

4. These projection layers appear larger after LoRA because they now include additional low-rank matrices:  
   `LoRA A` and `LoRA B`

5. The LoRA matrices are multiplied together and added to the original weight matrix, scaled by a factor `alpha`.

6. The alpha scaling is used to control `how much LoRA influences` the original model.

7. The LoRA rank `r` is much smaller than the original dimension. For example:
   - Original projection: `3072 × 3072`
   - LoRA: `A (3072 × 32)`, `B (32 × 3072)`, where `r = 32`

8. Multiplying `A` and `B` gives a matrix of size `3072 × 3072`, which matches the original weight matrix, so it can be added to it.

### `Why alpha scaling is needed`


The alpha scaling is used to control `how much LoRA influences` the original model.

Without scaling, the update from LoRA (`A × B`) might be too large and disturb the pretrained knowledge.

With scaling, we apply:

$$
W' = W + \frac{\alpha}{r} (A \times B)
$$

This ensures better balance between old knowledge and new learning.

### `Why rank is needed`

The rank constraint (r) `controls how much information LoRA can learn during fine-tuning.` This makes fine-tuning efficient while balancing model flexibility and resource usage.

- Instead of learning a full `3072 × 3072` matrix (very expensive), the update is factorized through a low-rank space of size `r`.

- This `rank constraint limits how much information can pass through the update.`

- **Small `r`** → `fewer patterns` learned (lightweight, fast, but limited capacity)  
- **Large `r`** → `more expressive` updates (better learning, but higher compute and memory cost)


### LoRA A and LoRA B Matrices


LoRA replaces a large update matrix with two smaller ones:

- **LoRA A**: Projects from high dimension → low dimension  
  `(3072 → 32)`

- **LoRA B**: Projects back from low dimension → high dimension  
  `(32 → 3072)`

So:

$$
\Delta W = A \times B
$$

### Why two matrices?

Instead of learning a full `3072 × 3072` matrix (very expensive),  
we learn two small matrices (cheap and efficient).

## Parameter-Efficient Fine-tuning

###  LoRA (Low-Rank Adaptation)

> This technique used for parameter-efficient fine-tuning of large language models. LoRA works by injecting small, trainable matrices into a pre-trained model's layers, rather than fine-tuning all of the model's parameters.

In [32]:
# LoRA config
# This defined which specific layers of the base model will get trained and
# it defines the architecture of the parameter-efficient modification.

lora_paramerters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES
)

In [42]:
lora_paramerters

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=128, target_modules={'down_proj', 'o_proj', 'k_proj', 'gate_proj', 'v_proj', 'up_proj', 'q_proj'}, exclude_modules=None, lora_alpha=256, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

## Supervised Fine-Tuning

In [43]:
# SFT config

train_parameters = SFTConfig(
    # ===== Output & Logging =====
    output_dir=PROJECT_RUN_NAME,        # where checkpoints are saved
    run_name=RUN_NAME,                  # run name (e.g., for wandb)
    report_to="wandb" if LOG_TO_WANDB else None,  # logging backend

    # ===== Training Duration =====
    num_train_epochs=EPOCHS,            # total training epochs
    max_steps=-1,                       # override epochs if > 0

    # ===== Batch Sizes =====
    per_device_train_batch_size=BATCH_SIZE,  # train batch per device
    per_device_eval_batch_size=1,            # eval batch per device
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,  # grad accumulation

    # ===== Optimization =====
    learning_rate=LEARNING_RATE,       # initial learning rate
    optim=OPTIMIZER,                   # optimizer type
    weight_decay=0.001,                # weight decay
    max_grad_norm=0.3,                 # gradient clipping

    # ===== Precision =====
    fp16=not use_bf16,                 # use fp16 if bf16 is off
    bf16=use_bf16,                     # use bf16 if available

    # ===== Scheduler =====
    lr_scheduler_type=LR_SCHEDULER_TYPE,  # LR scheduler type
    warmup_steps=WARMUP_RATIO,            # warmup steps

    # ===== Sequence Handling =====
    max_length=MAX_SEQUENCE_LENGTH,   # max sequence length
    group_by_length=True,             # speed up by grouping similar lengths

    # ===== Checkpointing =====
    save_strategy="steps",            # save by steps
    save_steps=SAVE_STEPS,            # save interval
    save_total_limit=10,              # max checkpoints to keep

    # ===== Evaluation =====
    eval_strategy="steps",            # eval by steps
    eval_steps=SAVE_STEPS,            # eval interval
    logging_steps=LOG_STEPS,          # logging frequency

    # ===== Hugging Face Hub =====
    push_to_hub=True,                 # upload model
    hub_model_id=HUB_MODEL_NAME,      # repo name
    hub_private_repo=True,            # private repo
    hub_strategy="every_save",        # push on each save
)

In [38]:
train_parameters

SFTConfig(output_dir='stream_price-fine-tuning-2026-04-30_11.01.12', do_train=False, do_eval=True, do_predict=False, eval_strategy=<IntervalStrategy.STEPS: 'steps'>, prediction_loss_only=False, per_device_train_batch_size=32, per_device_eval_batch_size=1, gradient_accumulation_steps=1, eval_accumulation_steps=None, eval_delay=0, torch_empty_cache_steps=None, learning_rate=0.0001, weight_decay=0.001, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, max_grad_norm=0.3, num_train_epochs=1, max_steps=-1, lr_scheduler_type=<SchedulerType.COSINE: 'cosine'>, lr_scheduler_kwargs=None, warmup_ratio=None, warmup_steps=0.01, log_level='passive', log_level_replica='warning', log_on_each_node=True, logging_dir=None, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_first_step=False, logging_steps=5, logging_nan_inf_filter=True, save_strategy=<SaveStrategy.STEPS: 'steps'>, save_steps=100, save_total_limit=10, enable_jit_checkpoint=False, save_on_each_node=False, save_only_model=False, 

## SFT Traniner

In [44]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_paramerters,
    args=train_parameters
)

Adding EOS to train dataset:   0%|          | 0/17600 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17600 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17600 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/2200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2200 [00:00<?, ? examples/s]

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss


In [ ]:
if LOG_TO_WANDB:
  wandb.finish()